In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "downstream_tasks").exists():
    REPO = Path("/home/jovyan/dpanc/GENA_LM/GENA_LM_expression_branch")

BENCHMARK_ROOT = Path("/home/jovyan/dpanc/benchmarking")
DATA = BENCHMARK_ROOT / "data"
GENA_ROOT = BENCHMARK_ROOT / "GENA_LM"
ALPHAGENOME_ROOT = BENCHMARK_ROOT / "AlphaGenome"

sys.path.insert(0, str(REPO / "downstream_tasks/expression_prediction/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions


In [2]:
split = "valid"  # valid or test
length_window = "1Mb"

pred_path = ALPHAGENOME_ROOT / "predictions" / f"alphagenome_benchmark_results_23072026_json14_len{length_window}_old_version" / f"alphagenome_predictions_{split}.tsv"
true_path = DATA / f"{split}_true_human.csv"
selected_targets_path = DATA / "selected_targets.csv"

pred = pd.read_csv(pred_path)
true = pd.read_csv(true_path)

if "gene_id" not in pred.columns:
    pred = pred.rename(columns={pred.columns[0]: "gene_id"})
if "gene_id" not in true.columns:
    true = true.rename(columns={true.columns[0]: "gene_id"})

print("pred:", pred.shape, pred_path)
print("true:", true.shape, true_path)
display(pred.head())
display(true.head())


pred: (2996, 15) /home/jovyan/dpanc/benchmarking/AlphaGenome/predictions/alphagenome_benchmark_results_23072026_json14_len1Mb_old_version/alphagenome_predictions_valid.tsv
true: (3038, 15) /home/jovyan/dpanc/benchmarking/data/valid_true_human.csv


,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000232604.1,0.000002,0.000002,0.000002,0.000004,0.000001,0.000003,0.000023,0.000029,4.310792e-07,0.000001,0.000002,0.000001,0.000003,0.000003
1,ENSG00000143942.4,0.117314,0.053410,0.096806,0.137785,0.110908,0.056703,0.230218,0.125281,1.490840e-01,0.245818,0.076579,0.140530,0.076112,0.044155
2,ENSG00000068912.13,0.086579,0.055255,0.090949,0.096522,0.104900,0.056730,0.138003,0.089296,1.381558e-01,0.185573,0.067477,0.104797,0.061413,0.048128
3,ENSG00000170634.12,0.011391,0.010544,0.011648,0.022922,0.006450,0.008360,0.015542,0.013297,6.242731e-03,0.007330,0.008002,0.005938,0.012283,0.004931
4,ENSG00000272156.1,0.000029,0.000024,0.000015,0.000023,0.000014,0.000026,0.000017,0.000057,3.344491e-06,0.000005,0.000015,0.000005,0.000021,0.000008


,gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
0,ENSG00000001617.11,1.079541,1.109003,2.621794,1.266682,3.045891,0.540609,2.526034,1.884014,3.266901,2.678770,1.053421,2.218972,1.311255,2.637604
1,ENSG00000002016.17,2.286177,2.088097,1.595354,1.543407,1.178615,2.598605,0.909459,2.293584,1.871749,1.846946,2.037829,2.124429,2.028351,1.698534
2,ENSG00000002549.12,3.037178,2.973722,3.893732,3.832327,3.886906,3.097516,3.550817,2.588759,2.945033,3.730597,2.922421,3.898263,3.879168,4.093605
3,ENSG00000002587.9,0.546619,0.256456,2.593309,0.687508,1.325394,0.918205,1.041303,0.168223,0.162103,0.006797,0.333568,0.025348,0.050543,0.348569
4,ENSG00000003393.14,2.238377,1.759425,2.019705,1.695560,1.892243,1.386311,1.919276,1.646933,2.870818,2.422747,1.912536,2.148924,2.786094,3.025538


In [3]:
def correlation_table(true_df, pred_df):
    true = true_df.set_index("gene_id")
    pred = pred_df.set_index("gene_id")

    common_genes = true.index.intersection(pred.index)
    common_cells = true.columns.intersection(pred.columns)

    true = true.loc[common_genes, common_cells]
    pred = pred.loc[common_genes, common_cells]

    rows = []
    for cell in common_cells:
        true_vec = true[cell].astype(float).values
        pred_vec = pred[cell].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 2 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            corr = np.nan
        else:
            corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        rows.append({"cell_type": cell, "corr_genes": corr})

    result = pd.DataFrame(rows)

    gene_corrs = []
    skipped_genes = []
    for gene in common_genes:
        true_vec = true.loc[gene].astype(float).values
        pred_vec = pred.loc[gene].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) < 4 or np.std(true_vec) == 0 or np.std(pred_vec) == 0:
            skipped_genes.append(gene)
            continue
        corr = np.corrcoef(true_vec, pred_vec)[0, 1]
        if np.isfinite(corr):
            gene_corrs.append(corr)
        else:
            skipped_genes.append(gene)

    mean_corr_cells = float(np.mean(gene_corrs)) if gene_corrs else np.nan
    result["corr_cells"] = mean_corr_cells

    mean_row = pd.DataFrame([{
        "cell_type": "mean",
        "corr_genes": result["corr_genes"].mean(),
        "corr_cells": mean_corr_cells,
    }])
    result = pd.concat([result, mean_row], ignore_index=True)

    return result, true.reset_index(), pred.reset_index(), common_genes, common_cells, skipped_genes


In [5]:
result, true_aligned, pred_aligned, common_genes, common_cells, skipped_genes = correlation_table(true, pred)
print("common genes:", len(common_genes))
print("common cells:", len(common_cells))
display(result)


common genes: 2996
common cells: 14


,cell_type,corr_genes,corr_cells
0,ENCFF035CWS,0.726192,0.238037
1,ENCFF083EOC,0.683851,0.238037
2,ENCFF123KIW,0.718014,0.238037
3,ENCFF236XOK,0.712445,0.238037
4,ENCFF242BWW,0.706183,0.238037
5,ENCFF329ENM,0.729286,0.238037
6,ENCFF361XCF,0.724472,0.238037
7,ENCFF494KRC,0.679962,0.238037
8,ENCFF602HCV,0.758450,0.238037
9,ENCFF660EXG,0.693429,0.238037


In [6]:
score_dict = score_predictions(true_aligned, pred_aligned, str(selected_targets_path), need_log=False)
deviation_r = score_dict.get("deviation_r", float("nan"))
print("deviation_r:", deviation_r)


deviation_r: 0.47286519392789694
